# MiniMax AutoResearch Chess Workshop

This notebook walks through the demo loop: baseline chess bot, local estimated Elo evaluator, constrained MiniMax patch, accept/reject, and replay.

## 0. Setup

Ensure the kernel runs from the repo root so `python -m autoresearch_chess.*` resolves. Safe to re-run.

In [ ]:
import os
from pathlib import Path

here = Path.cwd().resolve()
for candidate in [here, *here.parents]:
    if (candidate / "pyproject.toml").exists() and (candidate / "autoresearch_chess").is_dir():
        os.chdir(candidate)
        break
print("Working directory:", Path.cwd())

## 1. Hello, Tool Calling

Before we run the full agent loop, look at one round-trip on its own. We register one tool, send one user message, and see what MiniMax decides to do.

This cell uses `mock=True` so it works without an API key. Set `mock=False` if you have `MINIMAX_API_KEY` set and want to see the real model decide.

In [ ]:
import json
from autoresearch_chess.agent.tools import TOOLS
from autoresearch_chess.minimax_client import MiniMaxClient

# 1) One tool, in the JSON-schema format MiniMax (and OpenClaw) expect.
hello_tool = TOOLS[0]  # list_bot_files — the simplest of our four
tool_defs = [hello_tool.to_openai_format()]

print('=== TOOL DEFINITION (what we send to MiniMax) ===')
print(json.dumps(tool_defs[0], indent=2))

# 2) One round-trip. mock=True returns a scripted response so this works
#    without an API key; flip to mock=False for live MiniMax.
client = MiniMaxClient.from_environment(mock=True)
messages = [{'role': 'user', 'content': 'What chess bot files am I allowed to edit?'}]
response = client.chat_with_tools(messages, tools=tool_defs)

print()
print('=== MODEL RESPONSE (what MiniMax sent back) ===')
print(f"Provider: {response['meta']['provider']}")
print(f"Assistant text: {response['content']!r}")
print()
print('Tool calls:')
print(json.dumps(response['tool_calls'], indent=2))

## 2. Baseline

The starting bot is intentionally weak but legal. It uses `python-chess` for move legality and a shallow material-heavy search.

In [ ]:
!python -m autoresearch_chess.eval --out artifacts/notebook_eval.json

## 3. AutoResearch Loop

Five iterations of the OpenClaw-style agent driving MiniMax against the chess Elo evaluator. The eval is the only thing the agent cannot fake; only Elo-improving patches are accepted.

Editable surface (the agent may modify only these): `bot/evaluate.py`, `bot/search.py`, `bot/move_ordering.py`, `bot/config.py`. The evaluator, opponents, artifacts, tests, env files, and docs are forbidden.

In [ ]:
!python -m autoresearch_chess.loop --stage --mock-minimax --iterations 5

## 4. Replay

If live MiniMax or network access fails on stage, replay the captured artifact run.

In [ ]:
!python -m autoresearch_chess.replay --run artifacts/demo_replay

## 5. Reading the Agent's Trace

Each iteration writes a full ReAct trace to `iterations/NNN/agent_trace.jsonl` — one line per round, listing the tool calls the agent made. This is the agent's reasoning, made inspectable.

See `docs/openclaw_mapping.md` for how the agent modules map onto OpenClaw's gateway / context / react / tool-layer architecture.

In [ ]:
from pathlib import Path
import json

runs = sorted(Path('artifacts/runs').glob('*/iterations/001/agent_trace.jsonl'), key=lambda p: p.stat().st_mtime)
trace_path = runs[-1]
print('Reading', trace_path)
for line in trace_path.read_text(encoding='utf-8').splitlines():
    event = json.loads(line)
    calls = ', '.join(tc['name'] for tc in event['tool_calls']) or '<final message>'
    print(f"round {event['round']}: {calls}")

**Try this:** open the JSONL file in your editor and read the `arguments` and `result_preview` fields for each tool call. That is the agent's reasoning, made inspectable.

**Try this:** edit a tool description in `autoresearch_chess/agent/tools.py`, re-run the loop, and observe how the trace changes.